# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [4]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [6]:
#Functio to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [9]:
#Definition of the tables.
import pandas as pd

# Table and definitions sample
data = {'table': ['employees', 'salary', 'studies', 'departments', 'projects', 'assignments'],
        'definition': [
            'Employee information including name, employee_id, department_id, hire_date, position',
            'Salary details for each employee by year including employee_id, year, amount, currency',
            'Educational background including employee_id, institution_name, degree_type, field_of_study, graduation_year',
            'Department information including department_id, department_name, location, manager_id',
            'Project details including project_id, project_name, start_date, end_date, budget',
            'Employee project assignments including employee_id, project_id, role, hours_allocated'
        ]}
df = pd.DataFrame(data)
print(df)

         table                                         definition
0    employees  Employee information including name, employee_...
1       salary  Salary details for each employee by year inclu...
2      studies  Educational background including employee_id, ...
3  departments  Department information including department_id...
4     projects  Project details including project_id, project_...
5  assignments  Employee project assignments including employe...


In [10]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [11]:
print(text_tables)

employees: Employee information including name, employee_id, department_id, hire_date, position
salary: Salary details for each employee by year including employee_id, year, amount, currency
studies: Educational background including employee_id, institution_name, degree_type, field_of_study, graduation_year
departments: Department information including department_id, department_name, location, manager_id
projects: Project details including project_id, project_name, start_date, end_date, budget
assignments: Employee project assignments including employee_id, project_id, role, hours_allocated


In [12]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""


In [14]:
#Creating the prompt, with the user questions and the tables definitions.
pqt1 = prompt_question_tables.format(tables=text_tables, question="What is the average salary of employees in 2023?")

In [15]:
print(return_OAI(pqt1))

{
    "tables": ["employees", "salary"]
}


In [ ]:
pqt3 = prompt_question_tables.format(tables=text_tables,
                                     question="List all employees with a Master's degree who work on projects with a budget over 100000")

In [28]:
print(return_OAI(pqt3))

```json
{
    "tables": ["employees", "studies", "projects", "assignments"]
}
```


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [18]:
# Test Query 1: Simple single-table query
pqt_simple = prompt_question_tables.format(
    tables=text_tables, 
    question="What are the names of all departments?"
)
print("Query: What are the names of all departments?")
print("Response:")
print(return_OAI(pqt_simple))
print("\n" + "="*80 + "\n")

Query: What are the names of all departments?
Response:
```json
{
    "tables": ["departments"]
}
```




In [19]:
# Test Query 2: Two-table join
pqt_join = prompt_question_tables.format(
    tables=text_tables, 
    question="Which employees work in the Marketing department?"
)
print("Query: Which employees work in the Marketing department?")
print("Response:")
print(return_OAI(pqt_join))
print("\n" + "="*80 + "\n")

Query: Which employees work in the Marketing department?
Response:
{
    "tables": ["employees", "departments"]
}




In [20]:
# Test Query 3: Complex multi-table query
pqt_complex = prompt_question_tables.format(
    tables=text_tables, 
    question="Find employees who have a PhD, earn more than $80000, and are assigned to projects in the Technology department"
)
print("Query: Find employees who have a PhD, earn more than $80000, and are assigned to projects")
print("Response:")
print(return_OAI(pqt_complex))
print("\n" + "="*80 + "\n")

Query: Find employees who have a PhD, earn more than $80000, and are assigned to projects
Response:
{
    "tables": ["employees", "salary", "studies", "departments", "assignments"]
}




In [21]:
# Test Query 4: Aggregation query
pqt_agg = prompt_question_tables.format(
    tables=text_tables, 
    question="What is the total budget allocated across all projects?"
)
print("Query: What is the total budget allocated across all projects?")
print("Response:")
print(return_OAI(pqt_agg))
print("\n" + "="*80 + "\n")

Query: What is the total budget allocated across all projects?
Response:
```json
{
    "tables": ["projects", "assignments"]
}
```




In [22]:
# Test Query 5: Time-based query
pqt_time = prompt_question_tables.format(
    tables=text_tables, 
    question="Show the salary growth trend for employees hired before 2020"
)
print("Query: Show the salary growth trend for employees hired before 2020")
print("Response:")
print(return_OAI(pqt_time))
print("\n" + "="*80 + "\n")

Query: Show the salary growth trend for employees hired before 2020
Response:
```json
{
    "tables": ["employees", "salary"]
}
```




In [23]:
# Test Query 6: Ambiguous query (testing hallucination)
pqt_ambiguous = prompt_question_tables.format(
    tables=text_tables, 
    question="What is the performance rating of employees?"
)
print("Query: What is the performance rating of employees? (Note: performance rating is NOT in any table)")
print("Response:")
print(return_OAI(pqt_ambiguous))
print("\n" + "="*80 + "\n")

Query: What is the performance rating of employees? (Note: performance rating is NOT in any table)
Response:
```json
{
    "tables": ["employees"]
}
```




In [24]:
# Test Query 7: Query with unclear requirements
pqt_unclear = prompt_question_tables.format(
    tables=text_tables, 
    question="Give me information about employees"
)
print("Query: Give me information about employees (very vague)")
print("Response:")
print(return_OAI(pqt_unclear))
print("\n" + "="*80 + "\n")

Query: Give me information about employees (very vague)
Response:
```json
{
    "tables": ["employees"]
}
```


